# novel2epub — Server dịch zh → vi trên Colab / Kaggle

Notebook này dựng một **endpoint OpenAI-Compatible** (`/v1/chat/completions` + `/v1/models`) chạy trên GPU miễn phí
của Colab/Kaggle, rồi expose ra internet qua tunnel để dán thẳng vào **Cài đặt > Dịch API** và
**Cài đặt > AI biên tập** của novel2epub.

## Trước khi chạy

- **Colab**: `Runtime > Change runtime type > T4 GPU`.
- **Kaggle**: `Settings > Accelerator = GPU T4 x2` và **bật `Internet`** (cần xác minh số điện thoại).

## Cách dùng

1. Sửa **Cell CONFIG** (thường chỉ cần đổi `API_KEY`).
2. `Runtime > Run all`.
3. Cell **Verify** in ra khối cấu hình → copy vào Settings của novel2epub.
4. Để cell **Monitor** chạy trong lúc dịch (giữ session sống, in throughput).

## Giới hạn cần biết

| | Colab free | Kaggle |
| --- | --- | --- |
| GPU | 1× T4 15GB (sm75) | 2× T4 16GB (32GB, sm75) hoặc P100 |
| Session | ~12h, ngắt khi idle ~90 phút | 9h/session, 30h/tuần |

URL tunnel **đổi mỗi lần chạy lại** → phải cập nhật `base_url` trong Settings. Đây là công cụ chạy theo phiên
có người trông, không phải server 24/7.

Model tải về đĩa tạm nên **mất khi hết phiên**. Muốn giữ lại: xem cell *Lưu cache model cho phiên sau*
— Colab đặt `MODEL_CACHE = "drive"`, Kaggle attach model đã lưu qua `Add Input`.

> T4 là kiến trúc Turing (sm75): không bf16, không FlashAttention, không FP8 KV cache. Một số model đời mới nhất
> chưa có kernel Turing trong vLLM — khi đó đổi `ENGINE = "llamacpp"` là chạy được ngay.

## 1. CONFIG

In [ ]:
# ============================ CONFIG — chỉ cần sửa ở đây ============================

PRESET   = "auto"          # "auto" | id trong bảng PRESETS (cell kế tiếp)
ENGINE   = "auto"          # "auto" | "vllm" | "llamacpp"
TUNNEL   = "cloudflared"   # "cloudflared" (không cần tài khoản) | "ngrok" (cần token)

API_KEY  = "n2e-doi-chuoi-nay-di"   # dán đúng chuỗi này vào Settings của novel2epub

# --- Tinh chỉnh (mặc định hợp lý cho T4 16GB, để nguyên nếu không chắc) ---
MAX_MODEL_LEN  = 8192      # context; novel2epub gửi tối đa ~prompt_max_chars ký tự
MAX_NUM_SEQS   = 8         # số request song song → đặt translate.max_workers <= số này
USE_SHIM       = True      # shim chuẩn hoá request/response (xem cell Shim)
TEMP_CAP       = 0.35      # kẹp temperature: novel2epub mặc định 0.7, quá cao cho dịch
GPU_MEM_UTIL   = 0.90      # phần VRAM vLLM được phép chiếm

PORT_ENGINE    = 8000      # cổng vLLM / llama-server
PORT_SHIM      = 8010      # cổng shim (được tunnel ra ngoài khi USE_SHIM=True)
SERVED_NAME    = "novel2epub-zhvi"   # id model cố định → đổi preset không phải sửa Settings

VLLM_VERSION   = ""        # "" = bản mới nhất; ghim ví dụ "0.10.1" khi cần tái lập
HF_TOKEN       = ""        # chỉ cần cho repo gated

# --- Cache model giữa các phiên (xem cell "Lưu cache" ở cuối) ---
#   "auto"  : dùng dataset/model đã attach (Kaggle) hoặc Drive đã mount sẵn (Colab), không thì tải mới
#   "drive" : mount Google Drive và cache vào đó (Colab — có popup xác thực một lần)
#   "none"  : luôn tải mới
#   "/đường/dẫn" : thư mục cache tự chọn
MODEL_CACHE    = "auto"
DRIVE_CACHE_DIR = "/content/drive/MyDrive/novel2epub-models/hf"
MODEL_LOCAL_PATH = ""      # ép dùng model có sẵn: thư mục model hoặc file .gguf
                           # (vd Kaggle Model đã attach: "/kaggle/input/qwen-3/transformers/14b-awq/1")
# ===================================================================================

## 2. Nhận diện môi trường

Xác định Colab/Kaggle, số GPU, compute capability và nơi cache model.

In [ ]:
import json, os, re, shutil, subprocess, sys, time
from pathlib import Path

IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_URL_BASE"))
IS_COLAB = bool(os.environ.get("COLAB_GPU")) or "google.colab" in sys.modules

PROCS = {}          # tên -> subprocess.Popen (cell Stop dùng để dọn)
LOG_DIR = Path("/kaggle/working" if IS_KAGGLE else "/content")
LOG_DIR = LOG_DIR if LOG_DIR.exists() else Path.cwd()
BIN_DIR = LOG_DIR / "bin"
BIN_DIR.mkdir(parents=True, exist_ok=True)


def resolve_cache_dir():
    """Thư mục cache SỐNG QUA PHIÊN, hoặc None nếu phiên này tải mới."""
    if MODEL_CACHE == "none":
        return None
    if MODEL_CACHE not in ("auto", "drive", "none"):
        return Path(MODEL_CACHE)
    if not IS_COLAB:
        return None                       # Kaggle cache bằng input đã attach, xem CACHE_ROOTS
    drive_root = Path("/content/drive/MyDrive")
    if MODEL_CACHE == "drive" and not drive_root.exists():
        from google.colab import drive    # popup xác thực một lần
        drive.mount("/content/drive")
    if not drive_root.exists():
        return None                       # "auto": không tự mount để Run all không bị chặn
    return Path(DRIVE_CACHE_DIR)


def kaggle_input_roots():
    """Dataset/Model đã attach ở /kaggle/input có chứa model dùng được."""
    root = Path("/kaggle/input")
    if not root.exists():
        return []
    hits = []
    for child in root.iterdir():
        if not child.is_dir():
            continue
        if next(child.rglob("config.json"), None) or next(child.rglob("*.gguf"), None):
            hits.append(child)
    return hits


CACHE_DIR = resolve_cache_dir()
if CACHE_DIR:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(CACHE_DIR)   # tải một lần, phiên sau dùng lại
else:
    # Không có cache bền: dùng đĩa tạm (Kaggle chỉ có 20GB ở /kaggle/working nên tránh chỗ đó).
    tmp_home = Path("/kaggle/tmp/hf") if IS_KAGGLE else (
        Path("/content/hf") if IS_COLAB else Path.home() / ".cache/huggingface")
    tmp_home.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(tmp_home)

CACHE_ROOTS = ([CACHE_DIR] if CACHE_DIR else []) + kaggle_input_roots()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN


def gpu_info():
    """[{name, sm, vram_gb}] — thử torch trước, không có thì đọc nvidia-smi."""
    try:
        import torch
        if torch.cuda.is_available():
            out = []
            for i in range(torch.cuda.device_count()):
                p = torch.cuda.get_device_properties(i)
                out.append({"name": p.name, "sm": p.major * 10 + p.minor,
                            "vram_gb": round(p.total_memory / 1024 ** 3, 1)})
            return out
    except Exception:
        pass
    try:
        raw = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
             "--format=csv,noheader,nounits"], text=True)
    except Exception:
        return []
    out = []
    for line in raw.strip().splitlines():
        name, mem, cap = [x.strip() for x in line.split(",")]
        major, _, minor = cap.partition(".")
        out.append({"name": name, "sm": int(major) * 10 + int(minor or 0),
                    "vram_gb": round(int(mem) / 1024, 1)})
    return out


GPUS = gpu_info()
NUM_GPUS = len(GPUS)
SM = min((g["sm"] for g in GPUS), default=0)
VRAM_PER_GPU = min((g["vram_gb"] for g in GPUS), default=0.0)
VRAM_TOTAL = round(sum(g["vram_gb"] for g in GPUS), 1)
IS_TURING = SM == 75

PLATFORM = "Kaggle" if IS_KAGGLE else ("Colab" if IS_COLAB else "khác")
print(f"Nền tảng   : {PLATFORM}")
print(f"GPU        : {NUM_GPUS} × {GPUS[0]['name'] if GPUS else '(không có)'}")
print(f"VRAM       : {VRAM_PER_GPU} GB/GPU, tổng {VRAM_TOTAL} GB")
print(f"Compute cap: sm{SM}" + ("  ← Turing: fp16 only, không FlashAttention" if IS_TURING else ""))
print(f"HF_HOME    : {os.environ['HF_HOME']}"
      + ("  ← cache bền, phiên sau khỏi tải lại" if CACHE_DIR else "  ← đĩa tạm, mất khi hết phiên"))
for r in kaggle_input_roots():
    print(f"Input sẵn  : {r}")

if not GPUS:
    raise SystemExit(
        "Không thấy GPU. Colab: Runtime > Change runtime type > T4 GPU. "
        "Kaggle: Settings > Accelerator > GPU T4 x2, và bật Internet."
    )

## 3. Chọn model

Bảng preset + luật auto theo phần cứng. Repo được kiểm tra tồn tại trước khi tải.

In [ ]:
# Bảng model. `vram_gb` = trọng số sau lượng tử hoá (chưa tính KV cache).
PRESETS = {
    "qwen3-14b-awq": dict(
        engine="vllm", repo="Qwen/Qwen3-14B-AWQ", quant="awq", vram_gb=10.0, tp=1,
        thinking=True, note="Tiếng Trung rất mạnh, đã chạy ổn trên T4. Mặc định cho 1×T4."),
    "qwen2.5-14b-awq": dict(
        engine="vllm", repo="Qwen/Qwen2.5-14B-Instruct-AWQ", quant="awq", vram_gb=10.0, tp=1,
        thinking=False, note="Fallback an toàn nhất: không có thinking mode, kernel Turing đã chín."),
    "qwen3.5-9b-awq": dict(
        engine="vllm", repo="QuantTrio/Qwen3.5-9B-AWQ", quant="awq", vram_gb=6.5, tp=1,
        thinking=True, note="Đời mới (201 ngôn ngữ), nhẹ → thừa VRAM cho KV cache, batch lớn."),
    "qwen3.5-35b-a3b-awq": dict(
        engine="vllm", repo="QuantTrio/Qwen3.5-35B-A3B-AWQ", quant="awq", vram_gb=20.0, tp=2,
        thinking=True, note="Chỉ Kaggle T4×2. MoE chỉ kích hoạt ~3B/token nên vẫn nhanh."),
    "sailor2-20b-gguf": dict(
        engine="llamacpp", repo="bartowski/Sailor2-20B-Chat-GGUF", quant="Q4_K_M", vram_gb=12.0, tp=1,
        thinking=False, note="Nền Qwen2.5-14B, train chuyên Đông Nam Á → tiếng Việt tự nhiên nhất."),
    "sailor2-8b-gguf": dict(
        engine="llamacpp", repo="bartowski/Sailor2-8B-Chat-GGUF", quant="Q5_K_M", vram_gb=6.5, tp=1,
        thinking=False, note="Bản nhẹ của hướng Sailor2, hợp GPU nhỏ hoặc muốn nhanh."),
}

try:
    from huggingface_hub import hf_hub_download, list_repo_files, repo_exists
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import hf_hub_download, list_repo_files, repo_exists


def _fits(p):
    """Preset có vừa GPU đang có không (chừa ~2GB cho KV cache + overhead)."""
    if p["tp"] > NUM_GPUS:
        return False
    budget = VRAM_TOTAL if p["tp"] > 1 else VRAM_PER_GPU
    return p["vram_gb"] + 2.0 <= budget


def _candidates():
    """Thứ tự ưu tiên theo phần cứng, đã lọc theo ENGINE nếu người dùng ép."""
    if NUM_GPUS >= 2 and VRAM_TOTAL >= 30:
        order = ["qwen3.5-35b-a3b-awq", "qwen3-14b-awq", "sailor2-20b-gguf", "qwen2.5-14b-awq"]
    elif SM >= 80:
        order = ["qwen3.5-9b-awq", "qwen3-14b-awq", "sailor2-20b-gguf", "qwen2.5-14b-awq"]
    else:  # T4 đơn (sm75) — ưu tiên thứ đã biết chắc chạy được trên Turing
        order = ["qwen3-14b-awq", "qwen2.5-14b-awq", "sailor2-20b-gguf", "qwen3.5-9b-awq"]
    order += ["sailor2-8b-gguf"]
    seen, out = set(), []
    for pid in order:
        if pid in seen:
            continue
        seen.add(pid)
        p = PRESETS[pid]
        if ENGINE in ("vllm", "llamacpp") and p["engine"] != ENGINE:
            continue
        if not _fits(p):
            print(f"  bỏ qua {pid}: cần {p['vram_gb']}GB (tp={p['tp']}), máy có {VRAM_PER_GPU}GB/GPU × {NUM_GPUS}")
            continue
        out.append(pid)
    return out


def local_copy(preset):
    """Model đã nằm sẵn trong cache bền / input đã attach → trả đường dẫn, không thì None.

    Nhận layout cache của huggingface_hub (`models--org--repo/snapshots/<hash>`),
    tức là cả thư mục Drive lẫn dataset Kaggle tạo từ cell "Lưu cache".
    """
    repo = preset["repo"]
    tag = "models--" + repo.replace("/", "--")
    stem = repo.split("/")[-1].lower().removesuffix("-gguf")
    for root in CACHE_ROOTS:
        for hub_dir in root.rglob(tag):
            for snap in sorted((hub_dir / "snapshots").glob("*"), reverse=True):
                if preset["engine"] == "llamacpp":
                    hit = next((f for f in snap.rglob("*.gguf")
                                if preset["quant"].lower() in f.name.lower()), None)
                    if hit:
                        return hit
                elif (snap / "config.json").exists():
                    return snap
        if preset["engine"] == "llamacpp":
            hit = next((f for f in root.rglob("*.gguf")
                        if f.name.lower().startswith(stem)
                        and preset["quant"].lower() in f.name.lower()), None)
            if hit:
                return hit
        else:
            hit = next((c.parent for c in root.rglob("config.json")
                        if stem in c.parent.name.lower()), None)
            if hit:
                return hit
    return None


def _resolve():
    if PRESET != "auto":
        if PRESET not in PRESETS:
            raise SystemExit(f"PRESET {PRESET!r} không có. Chọn một trong: {', '.join(PRESETS)}")
        if not _fits(PRESETS[PRESET]):
            print(f"CẢNH BÁO: {PRESET} có thể không vừa VRAM — cứ thử, hỏng thì đổi preset nhẹ hơn.")
        return PRESET
    cands = _candidates()
    # Preset đã có sẵn trên đĩa được ưu tiên: khỏi tải lại chục GB.
    cands.sort(key=lambda pid: 0 if local_copy(PRESETS[pid]) else 1)
    for pid in cands:
        if local_copy(PRESETS[pid]):
            print(f"  {pid}: đã có sẵn trong cache")
            return pid
        # Repo cộng đồng có thể đổi tên/biến mất → xác nhận trước khi tải hàng chục GB.
        try:
            if repo_exists(PRESETS[pid]["repo"]):
                return pid
            print(f"  bỏ qua {pid}: repo {PRESETS[pid]['repo']} không truy cập được")
        except Exception as e:
            print(f"  bỏ qua {pid}: không kiểm tra được repo ({e})")
    raise SystemExit("Không preset nào chạy được với cấu hình hiện tại. Thử đặt PRESET thủ công.")


print("Chọn preset...")
PRESET_ID = _resolve()
P = PRESETS[PRESET_ID]
ENGINE_RESOLVED = P["engine"] if ENGINE == "auto" else ENGINE
TP = min(P["tp"], NUM_GPUS)

# Model đã có sẵn (MODEL_LOCAL_PATH ép tay, hoặc dò trong cache/input) → khỏi tải.
LOCAL_MODEL = Path(MODEL_LOCAL_PATH) if MODEL_LOCAL_PATH else local_copy(P)
if MODEL_LOCAL_PATH and not LOCAL_MODEL.exists():
    raise SystemExit(f"MODEL_LOCAL_PATH không tồn tại: {MODEL_LOCAL_PATH}")

# File GGUF: lấy đúng tên trong repo thay vì đoán (bartowski đặt tên theo từng bản).
GGUF_FILE = None
if ENGINE_RESOLVED == "llamacpp" and not (LOCAL_MODEL and LOCAL_MODEL.suffix == ".gguf"):
    files = [f for f in list_repo_files(P["repo"]) if f.endswith(".gguf")]
    match = [f for f in files if P["quant"].lower() in f.lower()]
    if not match:
        raise SystemExit(f"Không thấy bản {P['quant']} trong {P['repo']}. Có: {files[:10]}")
    GGUF_FILE = sorted(match, key=len)[0]

# Ước lượng KV cache để khuyên translate.max_workers cho đúng.
KV_MB_PER_TOKEN, EST_CONCURRENCY = None, MAX_NUM_SEQS
try:
    if LOCAL_MODEL and LOCAL_MODEL.is_dir() and (LOCAL_MODEL / "config.json").exists():
        cfg = json.load(open(LOCAL_MODEL / "config.json"))     # đọc offline được
    else:
        cfg = json.load(open(hf_hub_download(P["repo"], "config.json")))
    layers = cfg.get("num_hidden_layers") or cfg.get("n_layer")
    kv_heads = cfg.get("num_key_value_heads") or cfg.get("num_attention_heads")
    head_dim = cfg.get("head_dim") or (cfg["hidden_size"] // cfg["num_attention_heads"])
    KV_MB_PER_TOKEN = 2 * layers * kv_heads * head_dim * 2 / 1024 ** 2  # fp16 K+V
    free_gb = (VRAM_TOTAL if TP > 1 else VRAM_PER_GPU) * GPU_MEM_UTIL - P["vram_gb"]
    EST_CONCURRENCY = max(1, int(free_gb * 1024 / (KV_MB_PER_TOKEN * MAX_MODEL_LEN)))
except Exception:
    pass

print(f"\nPreset : {PRESET_ID} — {P['note']}")
print(f"Model  : {P['repo']}" + (f" :: {GGUF_FILE}" if GGUF_FILE else f" ({P['quant']})"))
print("Nguồn  : " + (f"đĩa sẵn có, không tải lại → {LOCAL_MODEL}" if LOCAL_MODEL
                     else "tải từ Hugging Face (xem cell Lưu cache để lần sau khỏi tải)"))
print(f"Engine : {ENGINE_RESOLVED}" + (f", tensor-parallel={TP}" if TP > 1 else ""))
if KV_MB_PER_TOKEN:
    print(f"KV cache: {KV_MB_PER_TOKEN:.3f} MB/token → ~{EST_CONCURRENCY} request song song ở context {MAX_MODEL_LEN}")
print(f"→ đặt translate.max_workers ≈ {min(MAX_NUM_SEQS, EST_CONCURRENCY)} trong novel2epub")

## 4. Cài đặt engine

vLLM: `pip install vllm` (có thể phải restart runtime một lần).
llama.cpp: binary CUDA dựng sẵn, không có thì build từ nguồn.

In [ ]:
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


def run_out(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)


t0 = time.time()
pip_install("huggingface_hub[hf_transfer]", "httpx", "requests")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
if USE_SHIM:
    pip_install("fastapi", "uvicorn[standard]")

MODEL_PATH = None      # vLLM: repo id | llama.cpp: đường dẫn file .gguf
LLAMA_SERVER = None    # đường dẫn binary llama-server
LLAMA_LIB_DIR = None

if ENGINE_RESOLVED == "vllm":
    spec = f"vllm=={VLLM_VERSION}" if VLLM_VERSION else "vllm"
    print(f"Cài {spec} (5–10 phút, pip sẽ thay torch của môi trường)...")
    pip_install(spec)
    probe = run_out([sys.executable, "-c", "import vllm, torch; print(vllm.__version__, torch.__version__)"])
    if probe.returncode != 0:
        print(probe.stderr[-2000:])
        raise SystemExit(
            "vLLM cài xong nhưng chưa import được — thường do torch vừa bị thay.\n"
            "→ Colab: Runtime > Restart session. Kaggle: Run > Restart & clear cell outputs.\n"
            "→ Rồi Run all lại từ đầu (lần này pip sẽ bỏ qua, chỉ mất vài giây)."
        )
    VLLM_VER, TORCH_VER = probe.stdout.split()
    print(f"vLLM {VLLM_VER} / torch {TORCH_VER}")
    # Có bản trên đĩa thì trỏ thẳng vào đó; không thì vLLM tự tải về HF_HOME.
    MODEL_PATH = str(LOCAL_MODEL) if LOCAL_MODEL else P["repo"]

else:
    # 1) Thử binary CUDA dựng sẵn từ release mới nhất (nhanh, ~30s).
    import requests
    try:
        rel = requests.get("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest", timeout=30).json()
        assets = [a["browser_download_url"] for a in rel.get("assets", [])]
        pick = next((u for u in assets if re.search(r"ubuntu.*cu(da)?.*x64.*\.zip$", u, re.I)), None)
        print(f"llama.cpp release {rel.get('tag_name')} — asset: {pick or '(không có bản ubuntu-cuda)'}")
    except Exception as e:
        pick, assets = None, []
        print(f"Không đọc được GitHub release ({e}) — sẽ build từ nguồn.")

    if pick:
        zip_path = BIN_DIR / "llamacpp.zip"
        subprocess.run(["curl", "-sL", "-o", str(zip_path), pick], check=True)
        shutil.unpack_archive(str(zip_path), str(BIN_DIR / "llamacpp"))
        found = list((BIN_DIR / "llamacpp").rglob("llama-server"))
        if found:
            LLAMA_SERVER = str(found[0])
            LLAMA_LIB_DIR = str(found[0].parent)
            os.chmod(LLAMA_SERVER, 0o755)
            env = {**os.environ, "LD_LIBRARY_PATH": LLAMA_LIB_DIR}
            if run_out([LLAMA_SERVER, "--version"], env=env).returncode != 0:
                print("Binary dựng sẵn không chạy được trên máy này → build từ nguồn.")
                LLAMA_SERVER = None

    # 2) Đường lui: build từ nguồn (~6 phút, chỉ target llama-server).
    if not LLAMA_SERVER:
        src = LOG_DIR / "llama.cpp"
        if not src.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/ggml-org/llama.cpp", str(src)], check=True)
        print(f"Build llama.cpp cho sm{SM} (~6 phút)...")
        subprocess.run(["cmake", "-B", str(src / "build"), "-S", str(src), "-DGGML_CUDA=ON",
                        f"-DCMAKE_CUDA_ARCHITECTURES={SM}", "-DLLAMA_CURL=OFF",
                        "-DCMAKE_BUILD_TYPE=Release"], check=True)
        subprocess.run(["cmake", "--build", str(src / "build"), "--config", "Release",
                        "-j", str(os.cpu_count() or 4), "--target", "llama-server"], check=True)
        found = list((src / "build").rglob("llama-server"))
        if not found:
            raise SystemExit("Build xong nhưng không thấy llama-server.")
        LLAMA_SERVER = str(found[0])
        LLAMA_LIB_DIR = str(found[0].parent)

    print(f"llama-server: {LLAMA_SERVER}")
    local_gguf = None
    if LOCAL_MODEL:
        local_gguf = LOCAL_MODEL if LOCAL_MODEL.suffix == ".gguf" else next(
            (f for f in LOCAL_MODEL.rglob("*.gguf") if P["quant"].lower() in f.name.lower()), None)
    if local_gguf:
        MODEL_PATH = str(local_gguf)
        print(f"Dùng GGUF có sẵn: {MODEL_PATH}")
    else:
        print(f"Tải {GGUF_FILE} ...")
        MODEL_PATH = hf_hub_download(P["repo"], GGUF_FILE)
    print(f"GGUF: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1024**3:.1f} GB)")

print(f"\nXong sau {time.time() - t0:.0f}s.")

## 5. Khởi động engine

Cờ được chọn theo phần cứng: `float16` + `XFORMERS` + `enforce-eager` trên Turing, và **tắt thinking mode** vì novel2epub chỉ gửi được một message `user`.

In [ ]:
import requests

ENGINE_URL = f"http://127.0.0.1:{PORT_ENGINE}"


def tail_log(path, n=40):
    try:
        return "".join(open(path, errors="replace").readlines()[-n:])
    except OSError:
        return "(chưa có log)"


def spawn(name, cmd, env=None):
    """Chạy nền, đổ stdout+stderr vào file log, trả (proc, log_path)."""
    log = LOG_DIR / f"{name}.log"
    fh = open(log, "wb")
    proc = subprocess.Popen(cmd, stdout=fh, stderr=subprocess.STDOUT, env=env or os.environ)
    PROCS[name] = proc
    return proc, log


def wait_ready(proc, log, url, timeout_s=1800, label="engine"):
    """Chờ GET {url}/v1/models trả 200. Model tải lần đầu có thể mất >10 phút."""
    headers = {"Authorization": f"Bearer {API_KEY}"}
    start, dots = time.time(), 0
    while time.time() - start < timeout_s:
        if proc.poll() is not None:
            print(tail_log(log, 60))
            raise SystemExit(
                f"{label} thoát với mã {proc.returncode}. Log ở trên.\n"
                + ("→ Lỗi kernel trên Turing (T4) là chuyện thường với model đời mới: "
                   "đặt ENGINE = \"llamacpp\" ở cell CONFIG rồi chạy lại từ đó.\n"
                   if ENGINE_RESOLVED == "vllm" else "")
                + "→ Hết VRAM thì giảm MAX_MODEL_LEN / MAX_NUM_SEQS, hoặc chọn preset nhẹ hơn."
            )
        try:
            r = requests.get(f"{url}/v1/models", headers=headers, timeout=5)
            if r.status_code == 200:
                print(f"\n{label} sẵn sàng sau {time.time() - start:.0f}s: {[m['id'] for m in r.json()['data']]}")
                return True
        except requests.RequestException:
            pass
        dots += 1
        print("." if dots % 10 else f" {time.time() - start:.0f}s ", end="", flush=True)
        time.sleep(3)
    raise SystemExit(f"{label} không sẵn sàng sau {timeout_s}s.\n{tail_log(log, 40)}")


def supports(help_text, flag):
    return flag in help_text


if ENGINE_RESOLVED == "vllm":
    help_text = run_out([sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--help"]).stdout

    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL_PATH,
           "--served-model-name", SERVED_NAME,
           "--host", "127.0.0.1", "--port", str(PORT_ENGINE),
           "--api-key", API_KEY,
           "--max-model-len", str(MAX_MODEL_LEN),
           "--max-num-seqs", str(MAX_NUM_SEQS),
           "--gpu-memory-utilization", str(GPU_MEM_UTIL)]
    if TP > 1:
        cmd += ["--tensor-parallel-size", str(TP)]
    if IS_TURING:
        # sm75 không có bf16; CUDA graph ăn thêm VRAM nên chạy eager cho rộng chỗ KV cache.
        cmd += ["--dtype", "float16", "--enforce-eager"]
        # Để auto thì vLLM chọn awq_marlin (chỉ sm80+); ép awq thường cho Turing.
        if P["quant"] == "awq":
            cmd += ["--quantization", "awq"]
    if supports(help_text, "--override-generation-config"):
        # novel2epub không gửi các tham số này → đặt mặc định ngay ở server.
        cmd += ["--override-generation-config",
                json.dumps({"top_p": 0.8, "top_k": 20, "repetition_penalty": 1.05})]
    if supports(help_text, "--disable-log-requests"):
        cmd += ["--disable-log-requests"]

    # Tắt thinking mode: client chỉ gửi 1 message user, không cách nào tắt từ phía app.
    if P["thinking"]:
        if supports(help_text, "--chat-template-kwargs"):
            cmd += ["--chat-template-kwargs", json.dumps({"enable_thinking": False})]
            print("Tắt thinking qua --chat-template-kwargs.")
        else:
            from transformers import AutoTokenizer
            tok = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
            tpl = getattr(tok, "chat_template", None)
            if tpl and "enable_thinking" in tpl:
                tpl_path = LOG_DIR / "chat_template_nothink.jinja"
                tpl_path.write_text("{%- set enable_thinking = false %}\n" + tpl, encoding="utf-8")
                cmd += ["--chat-template", str(tpl_path)]
                print(f"Tắt thinking bằng template vá: {tpl_path}")
            else:
                print("CẢNH BÁO: không vá được template — dựa vào shim để lọc <think>.")

    env = {**os.environ}
    if IS_TURING:
        env["VLLM_ATTENTION_BACKEND"] = "XFORMERS"   # FlashAttention cần sm80+
    print("Khởi động vLLM:")
    print("  " + " ".join(cmd))
    print()
    proc, log = spawn("engine", cmd, env)

else:
    help_text = run_out([LLAMA_SERVER, "--help"],
                        env={**os.environ, "LD_LIBRARY_PATH": LLAMA_LIB_DIR or ""}).stdout
    # llama.cpp chia -c cho -np slot, nên số slot bị VRAM còn lại quyết định.
    # Ước lượng thô 0.2 MB/token cho KV fp16 của model 14–20B; q8_0 giảm còn một nửa.
    kv_mb_tok = (KV_MB_PER_TOKEN or 0.20) / (2 if supports(help_text, "--cache-type-k") else 1)
    free_gb = max(0.5, VRAM_PER_GPU * 0.92 - P["vram_gb"] - 0.7)
    NP = max(1, min(MAX_NUM_SEQS, 4, int(free_gb * 1024 / (kv_mb_tok * MAX_MODEL_LEN))))
    print(f"VRAM còn ~{free_gb:.1f}GB sau trọng số → {NP} slot song song ở context {MAX_MODEL_LEN}")

    cmd = [LLAMA_SERVER, "-m", MODEL_PATH,
           "-ngl", "99", "-c", str(MAX_MODEL_LEN * NP), "-np", str(NP),
           "--host", "127.0.0.1", "--port", str(PORT_ENGINE),
           "--api-key", API_KEY]
    for flag, extra in [("--alias", [SERVED_NAME]), ("--cont-batching", []), ("--jinja", []),
                        ("--reasoning-budget", ["0"]), ("--no-webui", []),
                        ("--cache-type-k", ["q8_0"]), ("--cache-type-v", ["q8_0"])]:
        if supports(help_text, flag):
            cmd += [flag, *extra]
    env = {**os.environ, "LD_LIBRARY_PATH": LLAMA_LIB_DIR or ""}
    print("Khởi động llama-server:")
    print("  " + " ".join(cmd))
    print()
    proc, log = spawn("engine", cmd, env)
    EST_CONCURRENCY = NP

RECOMMENDED_WORKERS = max(1, min(MAX_NUM_SEQS, EST_CONCURRENCY))
wait_ready(proc, log, ENGINE_URL, label=ENGINE_RESOLVED)
print(tail_log(log, 8))

## 6. Shim OpenAI-compatible

Chuẩn hoá `model`, kẹp `temperature`, chèn `max_tokens`, lọc `<think>` ngay trên luồng SSE. Đặt `USE_SHIM = False` để bỏ qua.

In [ ]:
# Shim đứng trước engine để bù những gì novel2epub không gửi được:
# client chỉ gửi model + temperature + stream, không có extra_body (novel2epub/openai_client.py).
SHIM_CODE = r'''
import json, os, time
import httpx
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse

UPSTREAM = os.environ["SHIM_UPSTREAM"]
API_KEY = os.environ["SHIM_API_KEY"]
SERVED = os.environ["SHIM_SERVED_NAME"]
TEMP_CAP = float(os.environ.get("SHIM_TEMP_CAP", "0.35"))
MAX_TOKENS = int(os.environ.get("SHIM_MAX_TOKENS", "4096"))

app = FastAPI()
STATS = {"requests": 0, "errors": 0, "chunks_out": 0, "seconds": 0.0, "started": time.time()}
client = httpx.AsyncClient(base_url=UPSTREAM, timeout=httpx.Timeout(900.0, connect=15.0))


class ThinkFilter:
    """Bỏ khối <think>...</think> kể cả khi thẻ bị cắt ngang giữa hai SSE chunk."""

    OPEN, CLOSE = "<think>", "</think>"

    def __init__(self):
        self.buf = ""
        self.in_think = False
        self.emitted = False

    @staticmethod
    def _partial(buf, tag):
        for n in range(min(len(tag) - 1, len(buf)), 0, -1):
            if buf.endswith(tag[:n]):
                return n
        return 0

    def feed(self, text):
        self.buf += text
        out = []
        while self.buf:
            if self.in_think:
                i = self.buf.find(self.CLOSE)
                if i < 0:
                    keep = self._partial(self.buf, self.CLOSE)
                    self.buf = self.buf[len(self.buf) - keep:] if keep else ""
                    break
                self.buf = self.buf[i + len(self.CLOSE):]
                self.in_think = False
                continue
            i = self.buf.find(self.OPEN)
            if i < 0:
                keep = self._partial(self.buf, self.OPEN)
                out.append(self.buf[:len(self.buf) - keep] if keep else self.buf)
                self.buf = self.buf[len(self.buf) - keep:] if keep else ""
                break
            out.append(self.buf[:i])
            self.buf = self.buf[i + len(self.OPEN):]
            self.in_think = True
        text = "".join(out)
        if not self.emitted:
            text = text.lstrip()
            if text:
                self.emitted = True
        return text

    def flush(self):
        rest = "" if self.in_think else self.buf
        self.buf = ""
        return rest


def check_auth(request):
    got = (request.headers.get("authorization") or "").removeprefix("Bearer ").strip()
    if API_KEY and got != API_KEY:
        raise HTTPException(status_code=401, detail="api key sai")


def normalize(body):
    """Ép model về tên đang phục vụ, kẹp temperature, chèn max_tokens còn thiếu."""
    body["model"] = SERVED
    if body.get("temperature") is not None:
        body["temperature"] = min(float(body["temperature"]), TEMP_CAP)
    body.setdefault("max_tokens", MAX_TOKENS)
    body.setdefault("presence_penalty", 0.3)
    return body


def sse(obj):
    return ("data: " + json.dumps(obj, ensure_ascii=False) + "\n\n").encode()


def sse_delta(text):
    return sse({"id": "shim", "object": "chat.completion.chunk", "created": int(time.time()),
                "model": SERVED,
                "choices": [{"index": 0, "delta": {"content": text}, "finish_reason": None}]})


@app.get("/health")
async def health():
    return {"ok": True, **STATS}


@app.get("/_stats")
async def stats():
    return STATS


@app.get("/v1/models")
async def models(request: Request):
    check_auth(request)
    r = await client.get("/v1/models", headers={"Authorization": "Bearer " + API_KEY})
    if r.status_code != 200:
        raise HTTPException(status_code=r.status_code, detail=r.text[:500])
    return {"object": "list", "data": [{"id": SERVED, "object": "model", "owned_by": "novel2epub"}]}


@app.post("/v1/chat/completions")
async def chat(request: Request):
    check_auth(request)
    body = normalize(await request.json())
    headers = {"Authorization": "Bearer " + API_KEY, "Content-Type": "application/json"}
    started = time.time()
    STATS["requests"] += 1

    if not body.get("stream"):
        r = await client.post("/v1/chat/completions", json=body, headers=headers)
        if r.status_code != 200:
            STATS["errors"] += 1
            return JSONResponse(status_code=r.status_code, content={"error": r.text[:2000]})
        data = r.json()
        for ch in data.get("choices", []):
            msg = ch.get("message") or {}
            if isinstance(msg.get("content"), str):
                f = ThinkFilter()
                msg["content"] = (f.feed(msg["content"]) + f.flush()).strip()
        STATS["seconds"] += time.time() - started
        return JSONResponse(content=data)

    async def relay():
        filt = ThinkFilter()
        nchunk = 0
        try:
            async with client.stream("POST", "/v1/chat/completions", json=body, headers=headers) as up:
                if up.status_code != 200:
                    STATS["errors"] += 1
                    detail = (await up.aread()).decode("utf-8", "replace")[:2000]
                    yield sse({"error": detail})
                    return
                async for line in up.aiter_lines():
                    if not line.startswith("data:"):
                        continue
                    payload = line[5:].strip()
                    if payload == "[DONE]":
                        rest = filt.flush()
                        if rest:
                            yield sse_delta(rest)
                        yield b"data: [DONE]\n\n"
                        continue
                    try:
                        chunk = json.loads(payload)
                    except ValueError:
                        continue
                    choices = chunk.get("choices") or []
                    if choices:
                        delta = choices[0].get("delta") or {}
                        delta.pop("reasoning_content", None)
                        delta.pop("reasoning", None)
                        if isinstance(delta.get("content"), str):
                            kept = filt.feed(delta["content"])
                            if not kept:
                                continue
                            delta["content"] = kept
                            nchunk += 1
                    yield sse(chunk)
        finally:
            dt = time.time() - started
            STATS["chunks_out"] += nchunk
            STATS["seconds"] += dt
            rate = nchunk / dt if dt else 0.0
            print("[shim] " + str(nchunk) + " chunk / " + format(dt, ".1f") + "s = "
                  + format(rate, ".1f") + " tok/s", flush=True)

    return StreamingResponse(relay(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})
'''

PUBLIC_PORT = PORT_ENGINE
if USE_SHIM:
    (LOG_DIR / "shim.py").write_text(SHIM_CODE, encoding="utf-8")
    env = {**os.environ,
           "PYTHONPATH": str(LOG_DIR),
           "SHIM_UPSTREAM": ENGINE_URL,
           "SHIM_API_KEY": API_KEY,
           "SHIM_SERVED_NAME": SERVED_NAME,
           "SHIM_TEMP_CAP": str(TEMP_CAP),
           "SHIM_MAX_TOKENS": str(max(1024, MAX_MODEL_LEN // 2))}
    cmd = [sys.executable, "-m", "uvicorn", "shim:app", "--host", "0.0.0.0",
           "--port", str(PORT_SHIM), "--log-level", "warning"]
    proc_shim, log_shim = spawn("shim", cmd, env)
    wait_ready(proc_shim, log_shim, "http://127.0.0.1:" + str(PORT_SHIM), timeout_s=90, label="shim")
    PUBLIC_PORT = PORT_SHIM
else:
    print("USE_SHIM=False → tunnel trỏ thẳng vào engine (không lọc <think>, không kẹp temperature).")

## 7. Tunnel ra internet

In [ ]:
PUBLIC_URL = None

if TUNNEL == "cloudflared":
    cf = BIN_DIR / "cloudflared"
    if not cf.exists():
        subprocess.run(["curl", "-sL", "-o", str(cf),
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
                        "cloudflared-linux-amd64"], check=True)
        os.chmod(cf, 0o755)
    proc_tun, log_tun = spawn("tunnel", [str(cf), "tunnel", "--no-autoupdate",
                                         "--url", f"http://127.0.0.1:{PUBLIC_PORT}"])
    deadline = time.time() + 120
    while time.time() < deadline:
        if proc_tun.poll() is not None:
            print(tail_log(log_tun, 30))
            raise SystemExit("cloudflared thoát sớm. Thử lại, hoặc đổi TUNNEL = 'ngrok'.")
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tail_log(log_tun, 400))
        if m:
            PUBLIC_URL = m.group(0)
            break
        time.sleep(2)
    if not PUBLIC_URL:
        print(tail_log(log_tun, 30))
        raise SystemExit("Không lấy được URL trycloudflare sau 120s.")

elif TUNNEL == "ngrok":
    pip_install("pyngrok")
    from pyngrok import ngrok

    token = os.environ.get("NGROK_AUTHTOKEN", "")
    if not token and IS_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("NGROK_AUTHTOKEN")
        except Exception as e:
            print(f"Không đọc được Colab secret NGROK_AUTHTOKEN: {e}")
    if not token and IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
        except Exception as e:
            print(f"Không đọc được Kaggle secret NGROK_AUTHTOKEN: {e}")
    if not token:
        raise SystemExit(
            "Thiếu NGROK_AUTHTOKEN. Colab: biểu tượng chìa khoá bên trái > Add new secret "
            "(tên NGROK_AUTHTOKEN, bật Notebook access). Kaggle: Add-ons > Secrets. "
            "Hoặc dùng TUNNEL = 'cloudflared' (không cần tài khoản)."
        )
    ngrok.set_auth_token(token)
    PUBLIC_URL = ngrok.connect(PUBLIC_PORT, "http").public_url

else:
    raise SystemExit(f"TUNNEL {TUNNEL!r} không hợp lệ: chọn 'cloudflared' hoặc 'ngrok'.")

BASE_URL = PUBLIC_URL.rstrip("/") + "/v1"
print(f"Tunnel  : {PUBLIC_URL}  →  127.0.0.1:{PUBLIC_PORT}"
      f" ({'shim' if USE_SHIM else ENGINE_RESOLVED})")
print(f"base_url: {BASE_URL}")

## 8. Kiểm tra và lấy cấu hình dán vào novel2epub

In [ ]:
HAN_RE = re.compile(r"[一-鿿]")

SAMPLE_ZH = (
    "夜色如墨，青云宗后山的古松在寒风中低吟。林逸盘膝坐在断崖边，掌心翻转，"
    "一缕微弱的灵气缓缓凝聚。三年前他还是个连引气入体都做不到的废人，"
    "如今却已站在筑基的门槛前。「师兄，你当真要去闯那试炼塔？」"
    "少女的声音自身后传来，带着掩不住的担忧。"
    "林逸没有回头，只是淡淡道：「不去，我这一生就到此为止了。」"
)

PROMPT_TEMPLATE = (
    "Dịch đoạn văn tiếng Trung sau sang tiếng Việt. Giữ nguyên cách xuống dòng, "
    "văn phong truyện tiên hiệp, không thêm lời dẫn, chỉ trả về bản dịch.\n\n{text}"
)


def call_ai(prompt, temperature=0.7, base_url=None, timeout=600):
    """Gọi y hệt novel2epub: 1 message user, stream=True, ghép delta.content."""
    url = (base_url or BASE_URL).rstrip("/") + "/chat/completions"
    payload = {"model": SERVED_NAME,
               "messages": [{"role": "user", "content": prompt}],
               "temperature": temperature, "stream": True}
    started, first_at, parts = time.time(), None, []
    with requests.post(url, json=payload, timeout=timeout, stream=True,
                       headers={"Authorization": f"Bearer {API_KEY}",
                                "Content-Type": "application/json"}) as r:
        r.raise_for_status()
        for line in r.iter_lines():
            if not line:
                continue
            text = line.decode("utf-8", "replace").strip()
            if not text.startswith("data:"):
                continue
            payload_str = text[5:].strip()
            if not payload_str or payload_str == "[DONE]":
                continue
            try:
                chunk = json.loads(payload_str)
            except ValueError:
                continue
            if "error" in chunk:
                raise RuntimeError(chunk["error"])
            for choice in chunk.get("choices") or []:
                piece = (choice.get("delta") or {}).get("content")
                if piece:
                    first_at = first_at or time.time()
                    parts.append(piece)
    body = "".join(parts)
    return body, {"total_s": time.time() - started,
                  "first_token_s": (first_at - started) if first_at else None,
                  "chars": len(body)}


print("1) GET /v1/models qua tunnel — đúng đường mà Settings dùng để đổ dropdown")
r = requests.get(BASE_URL + "/models", headers={"Authorization": f"Bearer {API_KEY}"}, timeout=60)
r.raise_for_status()
print("   →", [m["id"] for m in r.json()["data"]])

print("\n2) Dịch thử một đoạn tiên hiệp (temperature 0.7 như mặc định của app)")
out, meta = call_ai(PROMPT_TEMPLATE.format(text=SAMPLE_ZH))
print("   " + out.replace("\n", "\n   "))
print(f"\n   {meta['chars']} ký tự trong {meta['total_s']:.1f}s"
      f" (token đầu sau {meta['first_token_s']:.1f}s)"
      f" ≈ {meta['chars'] / max(meta['total_s'], 0.01):.0f} ký tự/s")

problems = []
if "<think>" in out or "</think>" in out:
    problems.append("còn thẻ <think> — bật USE_SHIM hoặc kiểm tra template đã vá chưa")
if out.lstrip().startswith("```"):
    problems.append("mở đầu bằng fence Markdown (novel2epub tự bóc, nhưng nên chú ý)")
han = HAN_RE.findall(out)
if len(han) > 3:
    problems.append(f"còn {len(han)} ký tự Hán — dùng bước Clear Hán, hoặc đổi preset")
print("   " + ("Cảnh báo: " + "; ".join(problems) if problems else "Sạch: không <think>, không fence, không sót Hán."))

print(f"""
================= DÁN VÀO NOVEL2EPUB =================
Cài đặt > Dịch API            Cài đặt > AI biên tập
  base_url : {BASE_URL}
  api_key  : {API_KEY}
  model    : {SERVED_NAME}   (cả translation_model lẫn assistant_model)

Tham số nên đặt kèm:
  temperature        0.3        (shim kẹp trần {TEMP_CAP} dù app gửi cao hơn)
  timeout_seconds    600
  translate.max_workers  {RECOMMENDED_WORKERS}   (bằng số slot engine đang có)
  translate.prompt_max_chars  {min(7000, MAX_MODEL_LEN // 2)}   (khớp context {MAX_MODEL_LEN})

Hoặc trong novel2epub.yaml:
  defaults:
    ai:
      openai:
        base_url: "{BASE_URL}"
        api_key: "{API_KEY}"
        model: "{SERVED_NAME}"
    translate:
      type: openai
      openai:
        base_url: "{BASE_URL}"
        api_key: "{API_KEY}"
        model: "{SERVED_NAME}"

Kiểm tra lại từ máy chạy novel2epub:
  python scripts/check_openai_endpoint.py --base-url {BASE_URL} --api-key {API_KEY}
======================================================
URL này chết khi session Colab/Kaggle kết thúc — chạy lại notebook sẽ ra URL mới.""")

## 9. Lưu cache model cho phiên sau

Mặc định model tải về đĩa tạm và mất khi hết phiên. Cell này chỉ ra cách giữ lại: Drive trên Colab, input attach trên Kaggle.

In [ ]:
# Lưu model lại để phiên sau khỏi tải: đặt SAVE_CACHE = True rồi chạy cell này.
# Chỉ chạy được sau khi model đã tải xong (engine đã lên).
SAVE_CACHE = True

HUB_TAG = "models--" + P["repo"].replace("/", "--")


def dir_size_gb(path):
    path = Path(path)
    if path.is_file():
        return path.stat().st_size / 1024 ** 3
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file()) / 1024 ** 3


def model_files_on_disk():
    """Thư mục cache HF của model hiện tại (hoặc file .gguf) trong phiên này."""
    if LOCAL_MODEL:
        return LOCAL_MODEL
    hub = Path(os.environ["HF_HOME"]) / "hub" / HUB_TAG
    if hub.exists():
        return hub
    return next(Path(os.environ["HF_HOME"]).rglob(HUB_TAG), None)


def copy_cache(dst_root):
    src = model_files_on_disk()
    if src is None:
        raise SystemExit("Không tìm thấy model trên đĩa — chạy cell khởi động engine trước.")
    size = dir_size_gb(src)
    free = shutil.disk_usage(dst_root.parent if dst_root.exists() else "/").free / 1024 ** 3
    print(f"Nguồn : {src} ({size:.1f} GB)")
    print(f"Đích  : {dst_root} (còn trống {free:.1f} GB)")
    if free < size + 1:
        raise SystemExit(f"Không đủ chỗ: cần ~{size + 1:.1f} GB.")
    dst = dst_root / "hub" / HUB_TAG if src.is_dir() else dst_root / src.name
    started = time.time()
    if src.is_dir():
        # symlinks=False: cụ thể hoá blob thành file thật để bản copy tự đứng được.
        shutil.copytree(src, dst, dirs_exist_ok=True, symlinks=False)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"Đã chép {size:.1f} GB trong {(time.time() - started) / 60:.1f} phút → {dst}")
    return dst


if not SAVE_CACHE:
    print("SAVE_CACHE = False → bỏ qua.")

elif LOCAL_MODEL and CACHE_DIR and str(LOCAL_MODEL).startswith(str(CACHE_DIR)):
    print(f"Model đã nằm sẵn trong cache bền: {LOCAL_MODEL}")
    print("Phiên sau chỉ cần Run all — không tải lại gì.")

elif LOCAL_MODEL:
    print(f"Phiên này dùng model có sẵn tại {LOCAL_MODEL} — không cần chép lại.")
    if IS_KAGGLE:
        print("Giữ nguyên input đã attach là lần sau vẫn nhanh.")

elif CACHE_DIR:
    print(f"HF_HOME đã trỏ vào cache bền ({CACHE_DIR}) nên model vừa tải nằm luôn ở đó.")
    print(f"Dung lượng: {dir_size_gb(CACHE_DIR):.1f} GB. Phiên sau Run all là dùng lại ngay.")

elif IS_KAGGLE:
    # /kaggle/working (20GB) chỉ sống tiếp nếu bấm Save Version.
    dst = copy_cache(Path("/kaggle/working/model-cache"))
    print(f"""
Còn hai bước thủ công (Kaggle không cho tạo dataset từ trong session):
  1. Bấm 'Save Version' > 'Save & Run All' để output notebook này được giữ lại.
  2. Phiên sau: 'Add Input' > 'Your Work' > chọn output của notebook này.
     Notebook tự nhận model trong /kaggle/input và bỏ qua bước tải.

Cách gọn hơn nếu model đã có trên Kaggle: 'Add Input' > 'Models', tìm {P['repo'].split('/')[-1]},
attach rồi đặt MODEL_LOCAL_PATH trỏ vào thư mục đó — khỏi tốn 20GB output.""")

elif IS_COLAB:
    print("Chưa có cache bền. Hai lựa chọn:")
    print(f"  1. Đặt MODEL_CACHE = 'drive' ở cell CONFIG rồi Run all — model tải thẳng vào "
          f"{DRIVE_CACHE_DIR}, phiên sau khỏi tải (Drive free 15GB, đủ cho preset ~10GB).")
    print("  2. Chép ngay bây giờ: mount Drive rồi chạy lại cell này.")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        copy_cache(Path(DRIVE_CACHE_DIR))
        print("Phiên sau đặt MODEL_CACHE = 'drive' để dùng lại bản này.")
    except Exception as e:
        print(f"Chưa chép được: {e}")

else:
    print(f"Máy này không phải Colab/Kaggle — HF_HOME ({os.environ['HF_HOME']}) đã là cache bền.")

print("""
Ghi chú tốc độ: hf_transfer đang bật nên tải mới từ Hugging Face thường 2–5 phút cho
model ~10GB. Cache đáng giá nhất trên Kaggle (attach input = 0 giây, không tốn quota);
trên Colab, đọc từ Drive đôi khi chậm ngang tải mới — đo một lần rồi hãy quyết.""")

## 10. Monitor (để chạy trong lúc dịch)

In [ ]:
# Để cell này chạy trong lúc novel2epub dịch: vừa giữ session khỏi idle, vừa theo dõi tải.
# Dừng bằng nút ■ (Interrupt) — engine và tunnel vẫn chạy tiếp.
def vram_used():
    try:
        raw = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
            text=True)
        return " | ".join(f"{u.strip()}/{t.strip()}MB"
                          for u, t in (l.split(",") for l in raw.strip().splitlines()))
    except Exception:
        return "?"


started = time.time()
print(f"Theo dõi {PUBLIC_URL} — Ctrl+M I hoặc nút ■ để dừng.\n")
try:
    while True:
        for name, proc in PROCS.items():
            if proc.poll() is not None:
                print(f"\n{name} đã chết (mã {proc.returncode}):")
                print(tail_log(LOG_DIR / f"{name}.log", 25))
                raise SystemExit(f"{name} dừng — chạy lại cell tương ứng.")
        stats = {}
        if USE_SHIM:
            try:
                stats = requests.get(f"http://127.0.0.1:{PORT_SHIM}/_stats", timeout=5).json()
            except requests.RequestException:
                stats = {}
        line = (f"[{time.strftime('%H:%M:%S')}] uptime {(time.time() - started) / 60:.0f} phút"
                f" | VRAM {vram_used()}")
        if stats:
            line += (f" | {stats['requests']} request, {stats['errors']} lỗi,"
                     f" {stats['chunks_out']} chunk")
        print(line, flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("Đã dừng theo dõi. Engine và tunnel vẫn đang chạy.")

## 11. Chấm chất lượng zh → vi (tuỳ chọn)

In [ ]:
# Tuỳ chọn: chấm nhanh chất lượng trước khi dịch hàng loạt — đúng bước 1 trong
# docs/translation.md ("dịch thử vài chương đại diện"). Đổi preset rồi chạy lại
# cell này để so trực tiếp cùng một bộ mẫu.
SAMPLES = {
    "Hội thoại + xưng hô": (
        "「你以为凭你也配跟我谈条件？」中年男子冷笑一声，袖袍一挥。"
        "少年却毫不退让：「前辈说笑了，晚辈手里这块玉简，怕是宗主也想看一眼。」"
    ),
    "Tả cảnh": (
        "残阳沉入西山，血色霞光泼洒在万里荒原上。风卷起沙砾，"
        "打在破败的石碑上发出细碎的响声，碑上「镇魔」二字早已被岁月磨得模糊不清。"
    ),
    "Thuật ngữ tu luyện": (
        "他体内的灵力自丹田涌出，沿着奇经八脉运转三十六个周天，"
        "而后在膻中穴凝成一枚淡金色的气旋。筑基中期的瓶颈，终于松动了。"
    ),
}

for name, zh in SAMPLES.items():
    print("=" * 70)
    print(f"{name}\n原文: {zh}\n")
    try:
        vi, meta = call_ai(PROMPT_TEMPLATE.format(text=zh), temperature=0.3)
        print(f"{vi}\n[{meta['total_s']:.1f}s, {meta['chars']} ký tự,"
              f" {len(HAN_RE.findall(vi))} ký tự Hán còn sót]")
    except Exception as e:
        print(f"LỖI: {e}")
    print()

## 12. Dừng mọi thứ

In [ ]:
# Dọn dẹp: dừng tunnel, shim và engine để giải phóng VRAM mà không cần restart runtime.
for name in ("tunnel", "shim", "engine"):
    proc = PROCS.pop(name, None)
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except subprocess.TimeoutExpired:
            proc.kill()
        print(f"đã dừng {name}")
print("Xong. Chạy lại từ cell khởi động engine để dựng lại (URL tunnel sẽ khác).")